# Quick Start Guide - Microsoft Foundry

This notebook provides a hands-on introduction to Microsoft Foundry. You'll learn how to:
1. Initialize the AI Project client
2. List available models
3. Create a simple chat completion request
4. Create a basic AI agent
5. Handle basic error scenarios

## Prerequisites
- Completed environment setup from previous notebook
- Azure credentials configured
- **azure-ai-projects** package version 2.6.0 or greater (`azure-ai-projects>=2.6.0,<3.0.0`)
- **Foundry User role** assigned to your account for the Microsoft Foundry project
  - See [Microsoft Foundry RBAC documentation](https://learn.microsoft.com/azure/foundry/concepts/rbac-foundry) for more details on role assignments

## Import Required Libraries and Setup

In the next cell, we'll:
1. Import the necessary Azure SDK libraries for authentication and AI Projects
2. Import standard Python libraries for environment variables and JSON handling
3. Initialize a tenant-aware credential chain
   - The notebook first uses your logged-in Azure CLI credentials
   - If needed, it opens an interactive browser sign-in for the configured tenant

## 🔐 Authentication Setup

Before running the next cell, make sure you're authenticated with Azure CLI. 

* Open a terminal inside VSC (Visual Studio Code).
    * Run the following command in your terminal:

```
az login --use-device-code
```

* This will provide you with a device code and URL to authenticate in your browser to Azure.
    * Authenticate using the skillable Azure **Username** and **TAP**(Temporary Access Pass).
* Go back to the terminal and select the **default subscription.**

The Device Token will be used in this lab for:

* Remote development environments
* Systems without a default browser
* Corporate environments with strict security policies

* After successful authentication, you can proceed with the notebook cells below.

In [ ]:
# Import required libraries
from pathlib import Path
import os

from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from dotenv import load_dotenv

# Locate and load the nearest .env file, starting from the notebook working directory.
env_path = next(
    (directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()),
    None,
)
if env_path is None:
    raise FileNotFoundError(
        "Could not find a .env file. Complete Lab 00 and place .env in the repository root."
    )

load_dotenv(env_path)
print(f"📁 Environment loaded from: {env_path}")

tenant_id = os.getenv("TENANT_ID")
if not tenant_id:
    raise ValueError("TENANT_ID is missing from the root .env file.")

print(f"🔑 Using Tenant ID: {tenant_id}")

# The learner signs in with `az login --use-device-code` before running this cell.
credential = AzureCliCredential(tenant_id=tenant_id)
try:
    credential.get_token("https://ai.azure.com/.default")
except Exception as error:
    raise RuntimeError(
        "Azure CLI authentication failed. Run `az login --use-device-code --tenant "
        f"{tenant_id}` in the VS Code terminal, then rerun this cell."
    ) from error

print("✅ Successfully authenticated with Azure CLI!")

## Initialize AI Project Client

> **Note:** Before proceeding, ensure you:
> 1. Copy your `.env.example` file to `.env` from the root directory
> 2. Update the project endpoint in your `.env` file
> 3. Have a Foundry project already provisioned in Microsoft Foundry

You can find your project endpoint in [Microsoft Foundry](https://ai.azure.com) under **Manage > Project details**:

<img src="images/proj-conn-string.png" alt="Microsoft Foundry project endpoint location" width="600"/>


## Creating the AI Project Client

In the next cell, we'll create an AI Project client using the project endpoint from our `.env` file.
> **Note:** This example uses the synchronous client. For higher performance scenarios, you can also create an asynchronous client by importing `asyncio` and using the async methods from `AIProjectClient`.

The client will be used to:
- Connect to your Microsoft Foundry project using the project endpoint
- Authenticate using Azure credentials
- Enable making inference requests to your deployed models


In [ ]:
from dotenv import load_dotenv
from pathlib import Path

# Load environment variables from root directory
notebook_path = Path().absolute()
# Navigate to the root directory (up 3 levels: from Labs/Connection with AOAI/ to root)
root_dir = notebook_path.parent.parent.parent
env_path = root_dir / '.env'

print(f"📁 Looking for .env file at: {env_path}")
load_dotenv(env_path)

try:
    # Get the Microsoft Foundry project endpoint
    ai_foundry_project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    if not ai_foundry_project_endpoint:
        raise ValueError("AI_FOUNDRY_PROJECT_ENDPOINT not found in environment variables")
    
    print(f"🔗 Project Endpoint: {ai_foundry_project_endpoint}")
    
    # Create AIProjectClient as done in environment_setup (consistent with other notebooks)
    client = AIProjectClient(
        credential=credential,
        endpoint=ai_foundry_project_endpoint
    )
    print("✓ Successfully initialized AIProjectClient")
except Exception as e:
    print(f"× Error initializing client: {str(e)}")
    print("💡 Tip: Make sure your AI_FOUNDRY_PROJECT_ENDPOINT is set in the .env file at the root directory")
    print(f"💡 Expected .env location: {env_path}")

## Create a Simple Completion
Let's try a basic completion request:

Now that we have an authenticated client, let's use it to make a chat completion request.
The code below demonstrates how to:
1. Get an authenticated OpenAI client from the AI Project client
2. Use it to make a simple completion request

We'll use the MODEL_DEPLOYMENT_NAME from our `.env` file, making it easy to switch between different
deployed models without changing code. This could be an Azure OpenAI model, Microsoft model, or other providers
that support chat completions.

> Note: The OpenAI client uses the project-scoped `/openai/v1` endpoint provided by Microsoft Foundry.


In [ ]:
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

try:
    # Use the correct Azure AI Projects SDK pattern
    print("🔄 Getting OpenAI client from Azure AI Project...")
    print(f"🤖 Using model: {model_deployment_name}")
    
    # Get OpenAI client using the correct method
    openai_client = client.get_openai_client()
    
    # Create chat completion using OpenAI client pattern
    response = openai_client.chat.completions.create(
        model=model_deployment_name,
        messages=[
            {"role": "system", "content": "You are a helpful health assistant"},
            {"role": "user", "content": "How to be healthy in one sentence?"}
        ]
    )
    
    print("✅ Successfully created chat completion!")
    print(f"🤖 Assistant: {response.choices[0].message.content}")
    
except Exception as e:
    print(f"❌ An error occurred: {str(e)}")
    print("💡 Troubleshooting tips:")
    print("  - Ensure your Azure AI Project has OpenAI connections configured")
    print("  - Verify your MODEL_DEPLOYMENT_NAME is correctly deployed")
    print("  - Check that you have proper permissions to access the model")
    print("  - Make sure you're using the latest azure-ai-projects SDK version")

## Create a simple Agent

Using Microsoft Foundry Agent Service, we can create a simple prompt agent to answer health-related questions.

Let's explore Microsoft Foundry Agent Service, a powerful tool for building intelligent agents.

Microsoft Foundry Agent Service is a fully managed service that helps developers build, deploy, and scale AI agents
without managing infrastructure. It combines large language models with tools that allow agents to:
- Answer questions using RAG (Retrieval Augmented Generation)
- Perform actions through tool calling 
- Automate complex workflows

The code below demonstrates how to:
1. Create an agent with a code interpreter tool
2. Create an OpenAI conversation
3. Send a request for BMI analysis through the Responses API
4. Process the response and its file annotations
5. Save any generated visualizations to local files

The agent will use the model specified in our .env file (MODEL_DEPLOYMENT_NAME) and will have access
to a code interpreter tool for creating visualizations. This showcases how agents can combine
natural language understanding with computational capabilities.

> **Note:** Generated visualizations will be saved as PNG files in the same folder as this notebook.
 



In [ ]:
from azure.ai.projects.models import CodeInterpreterTool, PromptAgentDefinition

try:
    agent = client.agents.create_version(
        agent_name="bmi-calculator",
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions=(
                "You are a health analyst who calculates BMI using US metrics (pounds, feet/inches). "
                "Create clear visualizations and remind users that the result is general information, not medical advice."
            ),
            tools=[CodeInterpreterTool()],
        ),
        description="Calculates and visualizes BMI with Code Interpreter.",
    )
    print(f"✅ Created agent {agent.name}, version {agent.version}")

    conversation = openai_client.conversations.create()
    response = openai_client.responses.create(
        conversation=conversation.id,
        input=(
            "Calculate BMI for an average US female (5'4\", 130 lbs). "
            "Create a PNG visualization showing where this BMI falls on the standard BMI scale from 15 to 35. "
            "Include the standard BMI categories: Underweight, Normal, Overweight, and Obese."
        ),
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )
    print(f"🤖 Assistant: {response.output_text}")

    for output_item in response.output:
        if output_item.type != "message":
            continue
        for content_item in output_item.content:
            if content_item.type != "output_text":
                continue
            for annotation in content_item.annotations:
                if annotation.type != "container_file_citation":
                    continue
                file_content = openai_client.containers.files.content.retrieve(
                    file_id=annotation.file_id,
                    container_id=annotation.container_id,
                )
                file_name = annotation.filename or f"bmi_analysis_{annotation.file_id}.png"
                with open(file_name, "wb") as generated_file:
                    generated_file.write(file_content.read())
                print(f"📊 Visualization saved as: {file_name}")

    client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
    openai_client.conversations.delete(conversation_id=conversation.id)
    
except Exception as e:
    print(f"An error occurred: {str(e)}")